# Design of peptide library
### Goal: maximally diverse set of <96 peptides to test with LynD/PatD, PatG, and ArtGox enzymes

Peptides should be of the general form (based on JFT403 and CTK593 respectively):

XXXX(Het)APAYDG

or

WXXXXF(Het)APAYDG

AYDG = leaving group (can be also AYD); 
X = to be determined in workflow below; 
Het = heterocycle forming  

1. Do not include: Ser, Thr, Cys (heterocycle formation); Pro, Met 
2. Split amino acids into five categories
    - neutral: G, A, V, I, L, N, Q
    - neg. charged: D, E
    - aromatic: F, Y, W
    - basic: K, R, H
    - heterocycle forming (Het): Ser, Thr (with PatD), Cys (PatD+LynD) -> double check
3.  Make library of categories with maximal information
    - make permutations of the categories
    - remove peptides with excessive repeats (>2 times the same category in a peptide)
    - filter different levels of heterocyclization (C-ter Pro, 1, 2, and 3 additional heterocycles)
4. Sample peptides to get a set of maximally distant peptides
    - determine distance metric (physicochemical best)
    - set a seed sequence (CTK/JTF) from which to start sampling
    - calculate amount of peptides possible after step 3
    - figure out if exhaustive testing is possible or sampling is needed

In [43]:
from itertools import product as prod
from itertools import combinations_with_replacement as combination
from Bio.Align import substitution_matrices
from Bio import pairwise2
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime

# max amount of heterocycles, two are already in the design (SerXPro)
max_het = 2

category_no = range(5)
category_names = ['neut', 'neg', 'aromat', 'basic', 'het']

cat_dict = {'neut': ['G', 'A', 'V', 'I', 'L', 'N', 'Q', 'H'],
            'neg': ['D', 'E'],
            'aromat': ['F', 'Y', 'W'],
            'basic': ['K', 'R'],
            'het': ['S', 'T']
            } 

'''cat_dict = {'neut': ['G', 'A', 'V', 'I', 'L', 'N', 'Q', 'H'],
            'neg': ['D', 'E'],
            'aromat': ['F', 'Y', 'W'],
            'basic': ['K', 'R'],
            'het': ['C','S', 'T']
            } '''

'''cat_dict = {'neut': ['G', 'A', 'V', 'I', 'L', 'N', 'Q', 'H', 'F', 'Y', 'W'],
            'neg': ['D', 'E'],
            'basic': ['K', 'R'], 'het': ['S', 'T']
            }'''
# run_date = datetime.now()
# runID = run_date.strftime("%Y%m%d_%H-%M")
# logfile = open(runID[2:]+'_logfile.txt','w')

blosum62 = substitution_matrices.load("BLOSUM62")
rao_mat = substitution_matrices.load("RAO")


original_lib = ['PRYE', 'AFKG', 'YSFK', 'VADK',
'RYEW','TEYD','WEPL','FYAR'
'IRHE','EPHY','HDPG','RQYN'
'PDLE','HSLE','KYFT','SKYH'
'LWKP','GERF','LFDR','FHAE'
'WVEH','YEKV','SKID','YEDH'
'RDTW','SKPT','TQEH','DKSI'
'DWKS','ELWR','ERVF','RAYE'
'KWEA','HERF','RHAD','KVFK'
'VARH','KYWA','IKTY','FTKS']

original_lib_nohet = ['HRYE','AFKG','YAFK','FQKN',
                        'VADK','RYEW','VEYD','WEAL',
                        'FYAR','IRHE','EIHY','HDIG',
                        'RQYN','ADLE','HGLE','KYFV',
                        'IKYH','LWKG','GERF','LFDR',
                        'FHAE','WVEH','YEKV','VKID',
                        'YEDH','RDAW','IKVA','AQEH',
                        'DKAI','DWKV','ELWR','ERVF',
                        'RAYE','KWEA','HERF','RHAD',
                        'KVFK','VARH','KYWA','IKVY']

amino_dup = ['GG', 'AA', 'VV', 'II', 'LL', 'NN', 'QQ', 'HH', 'DD', 'EE', 'FF', 'YY', 'WW','KK', 'RR', 'SS', 'TT', 'CC']
        

In [44]:
def dist_mat(lib):
    '''calculate distance matrix of peptide library
    lib:    input list of peptides as strings
    '''
    score_mat = []
    dimension = lib.shape[0]

    for comb in prod(lib,lib):
        score_mat.append(pairwise2.align.globalds(comb[0], comb[1], blosum62, -10, -0.5, score_only = True))
    return(np.array(score_mat).reshape(dimension,dimension))


In [45]:
def dist_mat_sparse(lib):
    '''calculate distance matrix of peptide library
    lib:    input list of peptides as strings
    '''
    score_mat = []

    for comb in combination(lib, 2):
        score_mat.append(pairwise2.align.globalds(comb[0], comb[1], blosum62, -10, -0.5, score_only=True))
    return(np.array(score_mat))


In [46]:
def pep_lib_gen(c_lib):
    ''' create all possible peptides from the category based library'''
    library = []
    for concept_peptide in c_lib:
        cat_pep = []
        for aa_set in concept_peptide:
            cat_pep.append(cat_dict[aa_set])
        library.append(list(prod(*cat_pep)))
    library = np.array([''.join(item)+'TAP' for sublist in library for item in sublist], dtype=str)
    return(library.astype(str))

In [47]:
def sparse_to_full(sparse_vec):
    ''' convert sparse vector to a distance matrix'''
    dim = round((-1+np.sqrt(1+8*len(sparse_vec)))/2)
    mat = np.zeros((dim,dim))
    cnt = 0
    
    for i in range(dim):
        for j in range(i,dim):
            mat[i,j] = sparse_vec[cnt]
            mat[j,i] = sparse_vec[cnt]
            cnt += 1
    return(mat)

In [49]:
# make all possible combinations of categories
pep_len = 5
cat_library = np.array(list(prod(cat_dict, repeat=pep_len)))
print("Total amount of categides: " + str(cat_library.shape[0]))

# filter library according to point 3:
# no category >2 times, amount of heterocycles < max_het
m = []

'''for i, x in enumerate(cat_library):
    if max(np.unique(x, return_counts=True)[1]) > 2:  # filter duplicates
        m.append(True)
    elif sum(x == "het") > max_het - 2:  # filter according to heterocylces
        m.append(True)
    elif sum(x == "neg") + sum(x == "basic") < 1:  # filter solubility
        m.append(True)
    else:
        m.append(True)'''
        
for i, x in enumerate(cat_library):
    if max(np.unique(x, return_counts=True)[1]) > 2:  # filter duplicates
        m.append(False)
    elif sum(x == "het") > max_het - 2:  # filter according to heterocylces
        m.append(False)
    elif sum(x == "neg") + sum(x == "basic") < 1:  # filter solubility
        m.append(False)
    else:
        m.append(True)
m = np.array(m)

cat_library = cat_library[np.array(m)]
print("Amount of categides after filtering: " + str(cat_library.shape[0]))

Total amount of categides: 3125
Amount of categides after filtering: 600


In [50]:
# calculate amount of peptides total
tot = 0
for x in cat_library:
    peps = 1
    for i in x:
        peps = len(cat_dict[i]) * peps
    tot += peps
print('possible sequences in filtered library:\t' + str(tot))
print('total possible sequences:\t \t' + str(18**6))

possible sequences in filtered library:	259200
total possible sequences:	 	34012224


In [51]:
complete_lib = pep_lib_gen(cat_library)
print('Generated peptide library size: '+str(complete_lib.shape[0]))
print('Initiate distance matrix calculation')
#score_matrix = dist_mat(complete_lib[:11])
#score_matrix_sparse = dist_mat_sparse(complete_lib[:11])

# print('Saving distance matrix')
# np.save('neg_neut_basic_DM22', score_matrix)

Generated peptide library size: 259200
Initiate distance matrix calculation


In [52]:
# more filtering for homopolymers
def hasRepeatedChars(s):
    for x in amino_dup:
        if x in s:
            return(True)
    return(False)


filtered_lib = []

for seq in complete_lib:
    if hasRepeatedChars(seq) == False:
        filtered_lib.append(seq)
        
print('Length after filtering peptide seqs')
print(len(filtered_lib))      

Length after filtering peptide seqs
211320


In [53]:
np.save('220510_5mer_library', filtered_lib)